# Prueba de Detector de Video con Lógica Forense (OpenCV Puro)
Vamos a probar el algoritmo de OpenCV en los archivos de prueba.

In [22]:
import cv2
import numpy as np
from dataclasses import dataclass, field
from typing import Optional

# ─────────────────────────────────────────────────────────────────────────────
# Configuración de umbrales
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Config:
    # Módulo 1 – FFT
    fft_fake_threshold: float = 2.6
    fft_real_threshold: float = 4.5
    fft_fake_penalty:   float = 40.0
    fft_real_bonus:     float = 30.0

    # Módulo 2 – Laplaciano de ZONA DE BORDE (cuello/mandíbula)
    # Se analiza solo la franja inferior del rostro (20% inferior del ROI)
    # GANs: varianza baja ahí (<300) por el difuminado del cuello virtual
    # Rostros reales: varianza alta (>300) por poros, pelo y textura real
    laplacian_edge_zone_pct:  float = 0.20
    laplacian_blur_threshold: float = 300.0
    laplacian_penalty:        float = 20.0

    # Módulo 3 – Motion blur
    motion_blur_threshold: float = 150.0
    motion_bonus:          float = 15.0

    # Módulo 3b – Fondo estático (clon-avatar tipo Synthesia)
    # 270 frames ≈ 9s a 30fps
    static_bg_buffer_size: int   = 270
    static_bg_threshold:   float = 0.9
    static_bg_penalty:     float = 50.0

    # Módulo 4 – Clasificación con umbrales más separados
    score_authentic: float = 15.0   # < 15  → AUTÉNTICO
    score_deepfake:  float = 35.0   # > 35  → DEEPFAKE


# ─────────────────────────────────────────────────────────────────────────────
# Resultado por frame
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class FrameResult:
    fft_variance:      float = 0.0
    laplacian_score:   float = 0.0
    motion_blur_score: float = 0.0
    bg_variation:      float = 0.0
    total_score:       float = 0.0
    face_detected:     bool  = False
    label:             str   = "DESCONOCIDO"
    details:           list  = field(default_factory=list)


# ─────────────────────────────────────────────────────────────────────────────
# Módulo 1 – Transformada Rápida de Fourier
# ─────────────────────────────────────────────────────────────────────────────

def analyze_fft(face_roi: np.ndarray, cfg: Config) -> tuple[float, float, str]:
    gray      = cv2.cvtColor(face_roi, cv2.COLOR_BGR2GRAY) if face_roi.ndim == 3 else face_roi
    fft_shift = np.fft.fftshift(np.fft.fft2(gray))
    magnitude = np.log1p(np.abs(fft_shift))
    variance  = float(np.var(magnitude))

    if variance < cfg.fft_fake_threshold:
        return variance, cfg.fft_fake_penalty, \
            f"FFT aplanada ({variance:.2f} < {cfg.fft_fake_threshold}) -> IA detectada  +{cfg.fft_fake_penalty:.0f} pts"
    elif variance > cfg.fft_real_threshold:
        return variance, -cfg.fft_real_bonus, \
            f"FFT caotica ({variance:.2f} > {cfg.fft_real_threshold}) -> camara real  -{cfg.fft_real_bonus:.0f} pts"
    else:
        return variance, 0.0, \
            f"FFT ambigua ({variance:.2f}) -> zona gris  0 pts"


# ─────────────────────────────────────────────────────────────────────────────
# Módulo 2 – Laplaciano de ZONA DE BORDE (mandíbula → cuello)
#
# El error clásico de las GANs no está en los ojos ni en la nariz —
# está en la UNIÓN entre el cuello generado y la ropa/fondo real.
# Analizar el rostro completo oculta ese difuminado con la alta varianza
# del pelo y los ojos. Aquí solo miramos la franja inferior del ROI.
# ─────────────────────────────────────────────────────────────────────────────

def analyze_laplacian(face_roi: np.ndarray, cfg: Config) -> tuple[float, float, str]:
    gray = cv2.cvtColor(face_roi, cv2.COLOR_BGR2GRAY) if face_roi.ndim == 3 else face_roi
    h    = gray.shape[0]

    edge_start = int(h * (1.0 - cfg.laplacian_edge_zone_pct))
    edge_zone  = gray[edge_start:, :]

    if edge_zone.shape[0] < 4:
        return 0.0, 0.0, "Laplaciano: ROI demasiado pequeno  0 pts"

    variance = float(cv2.Laplacian(edge_zone, cv2.CV_64F).var())

    if variance < cfg.laplacian_blur_threshold:
        return variance, cfg.laplacian_penalty, \
            f"Borde cuello difuso ({variance:.0f} < {cfg.laplacian_blur_threshold}) -> GAN  +{cfg.laplacian_penalty:.0f} pts"
    else:
        return variance, 0.0, \
            f"Borde cuello nitido ({variance:.0f}) -> sin penalizacion  0 pts"


# ─────────────────────────────────────────────────────────────────────────────
# Módulo 3 – Motion Blur + Fondo Estático
# ─────────────────────────────────────────────────────────────────────────────

def analyze_motion(
    frame: np.ndarray,
    prev_frame: Optional[np.ndarray],
    bg_variation_buffer: list[float],
    has_face: bool,
    face_bbox: Optional[tuple],
    cfg: Config,
) -> tuple[float, float, float, str]:

    gray       = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    delta      = 0.0
    msgs: list[str] = []

    # Chequeo 1 – Temblor natural
    if blur_score < cfg.motion_blur_threshold and has_face:
        delta -= cfg.motion_bonus
        msgs.append(f"Camara temblorosa ({blur_score:.0f}) + cara -> natural  -{cfg.motion_bonus:.0f} pts")

    # Chequeo 2 – Fondo estático (solo cuando hay rostro)
    bg_var = 0.0
    if prev_frame is not None and has_face and face_bbox is not None:
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY) if prev_frame.ndim == 3 else prev_frame

        # Máscara que excluye el área del rostro + 20% de margen (hombros incluidos)
        # → medimos solo el fondo detrás de la persona
        mask = np.ones(gray.shape, dtype=np.uint8) * 255
        x, y, w, h = face_bbox
        pad_x = int(w * 0.2);  pad_y = int(h * 0.2)
        x0 = max(0, x - pad_x);          y0 = max(0, y - pad_y)
        x1 = min(gray.shape[1], x+w+pad_x); y1 = min(gray.shape[0], y+h+pad_y)
        mask[y0:y1, x0:x1] = 0

        diff   = cv2.absdiff(gray, prev_gray)
        bg_var = float(np.mean(diff[mask > 0]))
        bg_variation_buffer.append(bg_var)

    if has_face and len(bg_variation_buffer) >= cfg.static_bg_buffer_size:
        avg_bg = float(np.mean(bg_variation_buffer[-cfg.static_bg_buffer_size:]))
        if avg_bg < cfg.static_bg_threshold:
            delta += cfg.static_bg_penalty
            msgs.append(f"Fondo congelado ({avg_bg:.3f} px) -> CLON-AVATAR  +{cfg.static_bg_penalty:.0f} pts")

    if not msgs:
        msgs.append("Motion blur normal")

    return blur_score, bg_var, delta, " | ".join(msgs)


# ─────────────────────────────────────────────────────────────────────────────
# Módulo 4 – Clasificación
# ─────────────────────────────────────────────────────────────────────────────

def classify(score: float, cfg: Config) -> str:
    if score < cfg.score_authentic:
        return "AUTENTICO"
    elif score > cfg.score_deepfake:
        return "DEEPFAKE"
    return "SOSPECHOSO"


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline principal por frame
# ─────────────────────────────────────────────────────────────────────────────

def detect_deepfake_frame(
    frame: np.ndarray,
    face_detector,
    prev_frame: Optional[np.ndarray],
    bg_variation_buffer: list[float],
    cfg: Config,
) -> FrameResult:
    result = FrameResult()
    score  = 0.0
    gray   = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detección triple progresiva
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(60, 60))
    if len(faces) == 0:
        faces = face_detector.detectMultiScale(gray, scaleFactor=1.05, minNeighbors=2, minSize=(40, 40))
    if len(faces) == 0:
        faces = face_detector.detectMultiScale(
            cv2.equalizeHist(gray), scaleFactor=1.05, minNeighbors=2, minSize=(40, 40)
        )

    has_face  = len(faces) > 0
    face_bbox = None

    if has_face:
        result.face_detected = True
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        face_bbox   = (x, y, w, h)
        face_roi    = frame[y:y+h, x:x+w]

        # Módulo 1 – FFT
        fft_var, d1, m1 = analyze_fft(face_roi, cfg)
        result.fft_variance = fft_var
        score += d1
        result.details.append(m1)

        # Módulo 2 – Laplaciano zona de borde
        lap_var, d2, m2 = analyze_laplacian(face_roi, cfg)
        result.laplacian_score = lap_var
        score += d2
        result.details.append(m2)
    else:
        # Frame sin cara: score neutro (0) para no distorsionar promedios
        result.details.append("Sin rostro — frame neutro (score=0)")

    # Módulo 3 – Motion
    blur_s, bg_var, d3, m3 = analyze_motion(
        frame, prev_frame, bg_variation_buffer, has_face, face_bbox, cfg
    )
    result.motion_blur_score = blur_s
    result.bg_variation      = bg_var
    score += d3
    result.details.append(m3)

    result.total_score = max(0.0, score)
    result.label       = classify(result.total_score, cfg)
    return result


In [23]:
def probar_video_forense(video_path):
    print(f"\n{'='*50}")
    print(f"Analizando Video: {video_path}")
    print(f"{'='*50}\n")
    cfg = Config()
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    cap = cv2.VideoCapture(video_path)
    buf = []
    prev = None
    frame_count = 0
    faces_found = 0
    scores = []
    
    while cap.isOpened() and frame_count < 100:
        ret, frame = cap.read()
        if not ret: break
        res = detect_deepfake_frame(frame, detector, prev, buf, cfg)
        if res.face_detected: # Solo tomamos scores de frames con cara
            scores.append(res.total_score)
            faces_found += 1
        prev = frame.copy()
        frame_count += 1
        
    cap.release()
    if scores:
        avg_score = np.percentile(scores, 85)
        print(f"Frames evaluados: {frame_count} | Rostros hallados en {faces_found} frames ({(faces_found/max(1, frame_count))*100:.1f}%)")
        print(f"Puntaje Máximo Crítico (85° Percentil): {avg_score:.2f} pts")
        print("\nDetalles última evaluación:")
        for d in res.details: print(" >", d)
        print(f"\nVEREDICTO SUGERIDO: {classify(avg_score, cfg)}")
    else:
        print(f"Frames evaluados: {frame_count} | No se hallaron rostros en ningún frame.")


In [24]:
ruta_video_real = "backend/data/video real 6.mp4"
ruta_video_fake = "backend/data/video ia.mp4"
probar_video_forense(ruta_video_real)
probar_video_forense(ruta_video_fake)



Analizando Video: backend/data/video real 6.mp4

Frames evaluados: 100 | Rostros hallados en 35 frames (35.0%)
Puntaje Máximo Crítico (85° Percentil): 60.00 pts

Detalles última evaluación:
 > Sin rostro — frame neutro (score=0)
 > Motion blur normal

VEREDICTO SUGERIDO: DEEPFAKE

Analizando Video: backend/data/video ia.mp4

Frames evaluados: 100 | Rostros hallados en 99 frames (99.0%)
Puntaje Máximo Crítico (85° Percentil): 60.00 pts

Detalles última evaluación:
 > FFT aplanada (1.92 < 2.6) -> IA detectada  +40 pts
 > Borde cuello nitido (9220) -> sin penalizacion  0 pts
 > Motion blur normal

VEREDICTO SUGERIDO: DEEPFAKE
